In [1]:
import ee
import folium
import geemap
import geopandas as gpd
import json

In [2]:
ee.Initialize()

In [6]:
import ee
import geemap
import geopandas as gpd
import json
import os

# Inicialize o Earth Engine
ee.Initialize()

date1 = input("digite a primeira data, as datas precisão ser 'Mês e Dia' exemplo: 01-01")
date2 = input("digite a segunda data")
print(f"período {date1} e {date2}")
def obtem_ano(ano):
    # Carrega o shapefile e filtra para o PARNA Serra da Canastra
    pnsc = gpd.read_file(r"G:\Meu Drive\@EquipeGEO\zz.Bases\ICMBio\UC_Fed_nov_2020.shp")
    pnsc = pnsc.loc[pnsc["nome"] == "PARQUE NACIONAL DA SERRA DA CANASTRA"]
    pnsc_geojson = pnsc.to_json()

    # Converte o shapefile para um objeto Earth Engine
    pnsc_ee = ee.FeatureCollection(json.loads(pnsc_geojson))

    # Função para selecionar a coleção Landsat com base no ano
    def selecionarColecaoLandsat(ano):
        if ano >= 1984 and ano <= 1999:
            return "LANDSAT/LT05/C02/T1_L2"  # Landsat 5
        elif ano >= 1999 and ano <= 2012:
            return "LANDSAT/LE07/C02/T1_L2"  # Landsat 7
        elif ano >= 2013:
            return "LANDSAT/LC08/C02/T1_L2"  # Landsat 8
        else:
            raise ValueError("Ano fora do intervalo disponível para coleções Landsat")

    # Função para mascarar nuvens e sombras
    def maskLandsat(image):
        qa = image.select('QA_PIXEL')
        cloud = qa.bitwiseAnd(1 << 3).eq(0)
        shadow = qa.bitwiseAnd(1 << 5).eq(0)
        mask = cloud.And(shadow)
        return image.updateMask(mask)

    # Parâmetros de entrada
    data_inicio = f"{ano}-{date1}"
    data_fim = f"{ano}-{date2}"

    # Seleciona a coleção com base no ano
    colecao = selecionarColecaoLandsat(ano)

    # Filtra e aplica a máscara na coleção de imagens
    dataset = ee.ImageCollection(colecao) \
        .filterDate(data_inicio, data_fim) \
        .filterBounds(pnsc_ee) \
        .map(maskLandsat)

    # Seleciona as bandas para a visualização True Color e NBR
    if ano <= 2012:  # Landsat 5 e 7
        nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B4", "SR_B7"]).rename("NBR"))
    else:  # Landsat 8
        nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B5", "SR_B7"]).rename("NBR"))

    # Obtém o NBR mínimo (severidade máxima)
    nbrMin = nbr.min()

    # Calcula o centróide do PNSC
    centroid = pnsc_ee.geometry().centroid()

    # Inicializa o mapa centrado no centróide do PNSC
    Map = geemap.Map(center=(centroid.coordinates().get(1).getInfo(), centroid.coordinates().get(0).getInfo()), zoom=10)

    # Adiciona camadas ao mapa
    Map.addLayer(nbrMin.updateMask(nbrMin.lte(0)), 
             {"min": -1, "max": 0, "palette": ["orange"]}, 
             "NBR (Valores entre -1 e 0)")
    Map.addLayer(ee.Image().paint(pnsc_ee, 0, 2), {}, "Limite do PARNA Serra da Canastra")

    # Retorna o mapa atualizado
    return Map

# Exemplo de uso para um ano específico
mapa = obtem_ano(2022)
mapa


período 01-01 e 01-31


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [12]:
import ee
import geemap
import geopandas as gpd
import json
import os

# Inicialize o Earth Engine
ee.Initialize()

date1 = input("Digite a primeira data: ")
date2 = input("Digite a segunda data: ")
print(f"Período: {date1} e {date2}")

def obtem_ano(ano):
    # Carrega o shapefile do PARNA Serra da Canastra e filtra
    pnsc = gpd.read_file(r"G:\Meu Drive\@EquipeGEO\zz.Bases\ICMBio\UC_Fed_nov_2020.shp")
    pnsc = pnsc.loc[pnsc["nome"] == "PARQUE NACIONAL DA SERRA DA CANASTRA"]
    pnsc_geojson = pnsc.to_json()

    # Converte o shapefile para um objeto Earth Engine
    pnsc_ee = ee.FeatureCollection(json.loads(pnsc_geojson))

    # Carrega o shapefile de focos de calor
    focos = gpd.read_file(r"C:\Users\Diego\Downloads\721520d8-4f8e-5efa-8711-0d37f85a2724\focos_qmd_inpe_2022-01-01_2022-07-25_59.735122\focos_qmd_inpe_2022-01-01_2022-07-25_59.shp")
    focos = focos.loc[focos['Estado'] == 'MINAS GERAIS']
    focos_geojson = focos.to_json()

    # Converte o shapefile de focos de calor para um objeto Earth Engine
    focos_ee = ee.FeatureCollection(json.loads(focos_geojson))

    # Função para selecionar a coleção Landsat com base no ano
    def selecionarColecaoLandsat(ano):
        if ano >= 1984 and ano <= 1999:
            return "LANDSAT/LT05/C02/T1_L2"  # Landsat 5
        elif ano >= 1999 and ano <= 2012:
            return "LANDSAT/LE07/C02/T1_L2"  # Landsat 7
        elif ano >= 2013:
            return "LANDSAT/LC08/C02/T1_L2"  # Landsat 8
        else:
            raise ValueError("Ano fora do intervalo disponível para coleções Landsat")

    # Função para mascarar nuvens e sombras
    def maskLandsat(image):
        qa = image.select('QA_PIXEL')
        cloud = qa.bitwiseAnd(1 << 3).eq(0)
        shadow = qa.bitwiseAnd(1 << 5).eq(0)
        mask = cloud.And(shadow)
        return image.updateMask(mask)

    # Parâmetros de entrada
    data_inicio = f"{ano}-{date1}"
    data_fim = f"{ano}-{date2}"

    # Seleciona a coleção com base no ano
    colecao = selecionarColecaoLandsat(ano)

    # Filtra e aplica a máscara na coleção de imagens
    dataset = ee.ImageCollection(colecao) \
        .filterDate(data_inicio, data_fim) \
        .filterBounds(pnsc_ee) \
        .map(maskLandsat)

    # Seleciona as bandas para a visualização True Color e NBR
    if ano <= 2012:  # Landsat 5 e 7
        nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B4", "SR_B7"]).rename("NBR"))
    else:  # Landsat 8
        nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B5", "SR_B7"]).rename("NBR"))

    # Obtém o NBR mínimo (severidade máxima)
    nbrMin = nbr.min()

    # Calcula o centróide do PNSC
    centroid = pnsc_ee.geometry().centroid()

    # Inicializa o mapa centrado no centróide do PNSC
    Map = geemap.Map(center=(centroid.coordinates().get(1).getInfo(), centroid.coordinates().get(0).getInfo()), zoom=10)

    # Adiciona camadas ao mapa
    Map.addLayer(nbrMin.updateMask(nbrMin.lte(0)), 
                 {"min": -1, "max": 0, "palette": ["orange"]}, 
                 "NBR (Valores entre -1 e 0)")
    Map.addLayer(ee.Image().paint(pnsc_ee, 0, 2), {}, "Limite do PARNA Serra da Canastra")

    # Adiciona o shapefile de focos de calor com simbologia em ponto vermelho
    Map.addLayer(focos_ee, {"color": "red"}, "Focos de Calor")

    # Retorna o mapa atualizado
    return Map

# Exemplo de uso para um ano específico
mapa = obtem_ano(2022)
mapa


Período: 07-10 e 07-25


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [11]:
focos = gpd.read_file(r"C:\Users\Diego\Downloads\721520d8-4f8e-5efa-8711-0d37f85a2724\focos_qmd_inpe_2022-01-01_2022-07-25_59.735122\focos_qmd_inpe_2022-01-01_2022-07-25_59.shp")
focos = focos.loc[focos['Estado'] == 'MINAS GERAIS']
focos.head()

,DataHora,Satelite,Pais,Estado,Municipio,Bioma,DiaSemChuv,Precipitac,RiscoFogo,Latitude,Longitude,FRP,geometry
0,2022/04/17 15:29:00,AQUA_M-T,Brasil,RIO GRANDE DO NORTE,AREIA BRANCA,Caatinga,-999.0,4.5,0.0,-4.95011,-36.95779,34.5,POINT (-36.95779 -4.95011)
1,2022/04/17 17:05:00,AQUA_M-T,Brasil,MINAS GERAIS,FELIXLÂNDIA,Cerrado,1.0,0.0,1.0,-18.81404,-44.74966,22.5,POINT (-44.74966 -18.81404)
2,2022/04/17 17:05:00,AQUA_M-T,Brasil,MINAS GERAIS,ITAPAGIPE,Cerrado,4.0,0.0,0.9,-19.54416,-49.37927,12.5,POINT (-49.37927 -19.54416)
3,2022/04/17 17:05:00,AQUA_M-T,Brasil,SÃO PAULO,JARDINÓPOLIS,Cerrado,3.0,0.0,0.3,-20.96667,-47.83130,4.7,POINT (-47.83130 -20.96667)
4,2022/04/17 17:05:00,AQUA_M-T,Brasil,SÃO PAULO,CASA BRANCA,Cerrado,4.0,0.0,0.7,-21.81624,-46.98569,3.9,POINT (-46.98569 -21.81624)


In [53]:
pnsc = gpd.read_file(r"G:\Meu Drive\@EquipeGEO\zz.Bases\ICMBio\UC_Fed_nov_2020.shp")
pnsc = pnsc.loc[pnsc["nome"] == "PARQUE NACIONAL DA SERRA DA CANASTRA"]
pnsc_geojson = pnsc.to_json()

pnsc_ee = ee.FeatureCollection(json.loads(pnsc_geojson))

dataset = ee.ImageCollection("LANDSAT/LC08/C02/T1_TOA") \
    .filterDate("2020-01-01", "2020-12-31") \
    .filterBounds(pnsc_ee)

trueColor432 = dataset.select(["B4", "B3", "B2"])
trueColor432Vis = {
    "min": 0.0,
    "max": 0.4,
}

#Map.addLayer(trueColor432.median(), trueColor432Vis, "True Color (432)")
#Map.addLayer(ee.Image().paint(pnsc_ee, 0, 3), {}, "PARNA Serra da Canastra")

In [4]:
# Carregando e filtrando o shapefile
pnsc = gpd.read_file(r"G:\Meu Drive\@EquipeGEO\zz.Bases\ICMBio\UC_Fed_nov_2020.shp")
pnsc = pnsc.loc[pnsc["nome"] == "PARQUE NACIONAL DA SERRA DA CANASTRA"]
pnsc_geojson = pnsc.to_json()

# Convertendo para FeatureCollection do Earth Engine
pnsc_ee = ee.FeatureCollection(json.loads(pnsc_geojson))

# Filtrando a coleção de imagens Landsat
dataset = ee.ImageCollection("LANDSAT/LC08/C02/T1_TOA") \
    .filterDate("2020-01-01", "2020-12-31") \
    .filterBounds(pnsc_ee)

# Calculando o NBR
nbr = dataset.map(lambda image: image.normalizedDifference(["B5", "B7"]).rename("NBR"))

# Visualizando o NBR com as cores padrão
nbrVis = {
    "min": -1,
    "max": 1
}

Map.addLayer(nbr.median(), nbrVis, "NBR (Normalized Burn Ratio)")
Map.addLayer(ee.Image().paint(pnsc_ee, 0, 2), {}, "PARNA Serra da Canastra")

In [33]:
# Carregando e filtrando o shapefile
pnsc = gpd.read_file(r"G:\Meu Drive\@EquipeGEO\zz.Bases\ICMBio\UC_Fed_nov_2020.shp")
pnsc = pnsc.loc[pnsc["nome"] == "PARQUE NACIONAL DA SERRA DA CANASTRA"]
pnsc_geojson = pnsc.to_json()

# Convertendo para FeatureCollection do Earth Engine
pnsc_ee = ee.FeatureCollection(json.loads(pnsc_geojson))

# Filtrando a coleção de imagens Landsat
dataset = ee.ImageCollection("LANDSAT/LC08/C02/T1_TOA") \
    .filterDate("2020-01-01", "2020-12-31") \
    .filterBounds(pnsc_ee)

# Calculando o NBR
nbr = dataset.map(lambda image: image.normalizedDifference(["B5", "B7"]).rename("NBR"))

# Visualizando o NBR com as cores padrão
nbrVis = {
    "min": -1,
    "max": 1
}


dataset = ee.ImageCollection("LANDSAT/LC08/C02/T1_TOA") \
    .filterDate("2020-01-01", "2020-12-31") \
    .filterBounds(pnsc_ee)

trueColor432 = dataset.select(["B4", "B3", "B2"])
trueColor432Vis = {
    "min": 0.0,
    "max": 0.4,
}

Map.addLayer(trueColor432.median(), trueColor432Vis, "True Color (432)")
Map.addLayer(nbr.median(), nbrVis, "NBR (Normalized Burn Ratio)")
Map.addLayer(ee.Image().paint(pnsc_ee, 0, 2), {}, "PARNA Serra da Canastra")

In [72]:
# Carregando e filtrando o shapefile
pnsc = gpd.read_file(r"G:\Meu Drive\@EquipeGEO\zz.Bases\ICMBio\UC_Fed_nov_2020.shp")
pnsc = pnsc.loc[pnsc["nome"] == "PARQUE NACIONAL DA SERRA DA CANASTRA"]
pnsc_geojson = pnsc.to_json()
data_inicio = "2020-01-01"
data_fim = "2020-12-31"

# Convertendo para FeatureCollection do Earth Engine
pnsc_ee = ee.FeatureCollection(json.loads(pnsc_geojson))

dataset = ee.ImageCollection("LANDSAT/LC08/C02/T1_TOA") \
    .filterDate(data_inicio, data_fim) \
    .filterBounds(pnsc_ee)

trueColor432 = dataset.select(["B4", "B3", "B2"])
trueColor432Vis = {
    "min": 0.0,
    "max": 0.4,
}

# Função para mascarar nuvens e sombras usando o 'pixel_qa' da coleção Landsat 8
def maskL8sr(image):
    qa = image.select('QA_PIXEL')
    # Bits 3 e 5 são para nuvens e sombras de nuvens, respectivamente
    cloud = qa.bitwiseAnd(1 << 3).eq(0)
    shadow = qa.bitwiseAnd(1 << 5).eq(0)
    mask = cloud.And(shadow)
    return image.updateMask(mask)

# Filtrando a coleção de imagens Landsat e aplicando a máscara
dataset = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterDate(data_inicio, data_fim) \
    .filterBounds(pnsc_ee) \
    .map(maskL8sr)  # Aplicando a função de máscara

# Calculando o NBR
nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B5", "SR_B7"]).rename("NBR"))

# Selecionando o valor mínimo de NBR para cada pixel
nbrMin = nbr.min()

#Map.addLayer(trueColor432.median(), trueColor432Vis, "True Color (432)")
Map.addLayer(trueColor432.median(), trueColor432Vis, "True Color (432)")
Map.addLayer(nbrMin, nbrVis, "NBR Mínimo (Maior Severidade de Queimada)")
Map.addLayer(ee.Image().paint(pnsc_ee, 0, 2), {}, "PARNA Serra da Canastra")

In [5]:
Map.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
Map